
# Hybrid Retrieval & Reranking

**Day 3 — RAG & Agents · Practical 2 of 6 · Companion to the "Retrieval Techniques" deck**

> **Running in Google Colab:** works fine on the default **CPU runtime** — no GPU needed.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Run BM25 (sparse) retrieval alongside dense retrieval on the same legal corpus
2. Fuse both result lists with Reciprocal Rank Fusion (RRF)
3. Apply a cross-encoder reranker to the fused top-k results
4. See, on a concrete example, a case where dense retrieval alone misses an exact-term match
   that BM25 catches

## Why This Matters for a Law Firm

Legal text is full of exact terms that matter precisely — defined terms, section numbers,
statute citations, Latin phrases. This notebook shows exactly where pure semantic (dense)
retrieval falls short on those terms, and how hybrid retrieval plus reranking closes that gap.

## Notebook Workflow

```mermaid
flowchart TD
    A["Query"] --> B["Dense Retrieval\n(embeddings)"]
    A --> C["Sparse Retrieval\n(BM25)"]
    B --> D["Reciprocal Rank\nFusion"]
    C --> D
    D --> E["Top-K fused\ncandidates"]
    E --> F["Cross-Encoder\nReranker"]
    F --> G["Final ranked\nresults"]



## Section 1 — Setup


In [ ]:

%pip install -q rank_bm25 sentence-transformers

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

print("BM25 and sentence-transformers ready.")



## Section 2 — The Legal Corpus

Same style of corpus as the previous notebook, with one document deliberately containing a
specific defined term ("Force Majeure Event") that a dense embedding model may not weight as
precisely as an exact keyword match would.


In [ ]:

legal_documents = [
    "Section 7.1 Indemnification. The Contractor shall indemnify, defend, and hold harmless the Client from any claims arising from gross negligence or willful misconduct.",
    "Section 9.2 Termination for Convenience. Either party may terminate this Agreement without cause upon sixty (60) days' prior written notice.",
    "Section 12.1 Confidentiality. The Receiving Party shall maintain all Confidential Information in strict confidence and shall not disclose it to any third party.",
    "Section 14.3 Limitation of Liability. In no event shall either party's total liability exceed the total fees paid in the twelve (12) months preceding the claim.",
    "Section 16.1 Governing Law. This Agreement shall be governed by the laws of the State of Delaware, without regard to conflict of laws principles.",
    "Section 18.1 Force Majeure. A Force Majeure Event includes acts of God, natural disasters, war, or governmental action, excusing delay if prompt notice is given.",
    "Section 20.4 Assignment. Neither party may assign this Agreement without the prior written consent of the other party, except to a successor by merger.",
    "Section 22.1 Notices. All notices under this Agreement shall be delivered in writing to the addresses set forth in Exhibit A.",
]

print(f"Corpus size: {len(legal_documents)} documents")



## Section 3 — Dense Retrieval

Standard embedding-based retrieval, same mechanism as the previous notebook.


In [ ]:

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
doc_embeddings = embedding_model.encode(legal_documents)

def dense_retrieve(query, k=5):
    query_embedding = embedding_model.encode([query])[0]
    # Cosine similarity between the query and every document
    scores = doc_embeddings @ query_embedding / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(query_embedding)
    )
    ranked_indices = np.argsort(-scores)[:k]
    return [(i, legal_documents[i], scores[i]) for i in ranked_indices]

query = "What counts as a Force Majeure Event?"
print("DENSE RETRIEVAL results:\n")
for idx, text, score in dense_retrieve(query, k=3):
    print(f"  [{idx}] score={score:.4f}  {text[:90]}...")



## Section 4 — Sparse Retrieval (BM25)

BM25 matches on exact term overlap. Watch how it ranks the Force Majeure clause for this
specific query.


In [ ]:

tokenized_corpus = [doc.lower().split() for doc in legal_documents]
bm25 = BM25Okapi(tokenized_corpus)

def sparse_retrieve(query, k=5):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    ranked_indices = np.argsort(-scores)[:k]
    return [(i, legal_documents[i], scores[i]) for i in ranked_indices]

print("SPARSE (BM25) RETRIEVAL results:\n")
for idx, text, score in sparse_retrieve(query, k=3):
    print(f"  [{idx}] score={score:.4f}  {text[:90]}...")



## Section 5 — Reciprocal Rank Fusion (RRF)

RRF combines both rankings based on RANK POSITION, not raw score -- avoiding the problem of
BM25 scores and cosine-similarity scores living on incomparable scales.

RRF formula: for each document, `score = sum(1 / (rank_constant + rank))` across every ranked
list it appears in. A document ranked highly in EITHER list gets a strong combined score.


In [ ]:

def reciprocal_rank_fusion(dense_results, sparse_results, rank_constant=60, k=5):
    fused_scores = {}

    for rank, (idx, text, _) in enumerate(dense_results):
        fused_scores.setdefault(idx, {"text": text, "score": 0.0})
        fused_scores[idx]["score"] += 1 / (rank_constant + rank)

    for rank, (idx, text, _) in enumerate(sparse_results):
        fused_scores.setdefault(idx, {"text": text, "score": 0.0})
        fused_scores[idx]["score"] += 1 / (rank_constant + rank)

    ranked = sorted(fused_scores.items(), key=lambda item: -item[1]["score"])
    return [(idx, data["text"], data["score"]) for idx, data in ranked[:k]]

dense_results = dense_retrieve(query, k=5)
sparse_results = sparse_retrieve(query, k=5)
fused_results = reciprocal_rank_fusion(dense_results, sparse_results, k=5)

print("HYBRID (RRF-FUSED) results:\n")
for idx, text, score in fused_results:
    print(f"  [{idx}] rrf_score={score:.4f}  {text[:90]}...")



## Section 6 — Cross-Encoder Reranking

The fused top-k candidates get one more pass: a cross-encoder jointly encodes the query AND
each candidate document together, producing a more precise relevance score than either
retrieval method alone -- the deck's "retrieve broad, rerank narrow" pattern.


In [ ]:

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

candidate_texts = [text for _, text, _ in fused_results]
pairs = [[query, text] for text in candidate_texts]
rerank_scores = reranker.predict(pairs)

reranked = sorted(zip(fused_results, rerank_scores), key=lambda x: -x[1])

print("RERANKED (final) results:\n")
for (idx, text, _), rerank_score in reranked:
    print(f"  [{idx}] rerank_score={rerank_score:.4f}  {text[:90]}...")



## Section 7 — Comparing All Four Stages Side by Side

Put dense-only, sparse-only, hybrid, and reranked results next to each other for the same query,
to see the full pipeline's effect in one view.


In [ ]:

print(f"Query: {query!r}\n")

print("Top-1 result by method:")
print(f"  Dense only:  [{dense_results[0][0]}] {dense_results[0][1][:70]}...")
print(f"  Sparse only: [{sparse_results[0][0]}] {sparse_results[0][1][:70]}...")
print(f"  Hybrid RRF:  [{fused_results[0][0]}] {fused_results[0][1][:70]}...")
print(f"  Reranked:    [{reranked[0][0][0]}] {reranked[0][0][1][:70]}...")



## Section 8 — Try It Yourself

Try a query using an exact defined term or section number, and compare how dense-only search
handles it versus BM25/hybrid.


In [ ]:

your_query = "What does Section 20.4 say about assignment?"

your_dense = dense_retrieve(your_query, k=3)
your_sparse = sparse_retrieve(your_query, k=3)

print("DENSE top result:", your_dense[0][1][:90])
print("SPARSE top result:", your_sparse[0][1][:90])



## Key Takeaways

1. **BM25 and dense retrieval genuinely disagree** on some queries -- exact terms and section
   numbers favor BM25, paraphrased/semantic queries favor dense retrieval. Hybrid retrieval
   doesn't have to choose.
2. **RRF fuses by rank, not by score** -- sidestepping the problem that BM25 scores and cosine
   similarities aren't on the same numeric scale.
3. **The cross-encoder reranker** re-scores using genuine query-document joint attention, not
   just vector similarity -- more accurate, but deliberately only applied to the small fused
   top-k, not the whole corpus, per the deck's two-stage pattern.

**Next up:** the *RAGAS Evaluation & CRAG-Style Evaluator* notebook — measuring and
self-correcting this pipeline.
